In [1]:
from helpers import get_device
import torch
from transformers import AutoTokenizer, GPT2LMHeadModel

In [2]:
# Load pre-trained GPT-2 (124M parameters)
tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

In [3]:
device = get_device()
model = model.to(device)
print(f"Model loaded on {device}")

Model loaded on mps


In [4]:
# Basic greedy generation, always picks the highest probability token
prompt = "In 36 months, the cheapest place to put AI will be"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

outputs = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,  # Greedy decoding
    pad_token_id=tokenizer.eos_token_id,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In 36 months, the cheapest place to put AI will be in the US.

The US is the world's largest economy, with a population of about 1.5 billion.

The US is also the world's largest economy, with a population of about 1.5 billion.

The


In [5]:
# Temperature sampling, controls randomness by scaling logits before softmax
inputs = tokenizer(prompt, return_tensors="pt").to(device)

for temp in [0.3, 0.7, 1.0, 1.5]:
    print(f"\n--- Temperature: {temp} ---")
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,
        temperature=temp,
        pad_token_id=tokenizer.eos_token_id,
    )
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))


--- Temperature: 0.3 ---
In 36 months, the cheapest place to put AI will be in the UK.

The UK has the highest rate of AI in the world, with more than a million jobs in the UK.

The UK has the highest rate of AI in the world, with more than a million jobs in the

--- Temperature: 0.7 ---
In 36 months, the cheapest place to put AI will be in the U.S.

The average price per square foot is $5.45

In 36 months, the cheapest place to put AI will be in the U.S. The average price per square foot is $5.45

--- Temperature: 1.0 ---
In 36 months, the cheapest place to put AI will be near the sea. When you first fly, it will be flying below what you're looking for. The easiest way is to think about that, but then start the same thing over and over again. AI is doing your flying. People get frustrated trying

--- Temperature: 1.5 ---
In 36 months, the cheapest place to put AI will be at my favourite social platform;

At least, every time. AI startups are still trying a novel way of generating rev

In [6]:
# Top-k sampling, only samples from the top k most likely tokens
inputs = tokenizer(prompt, return_tensors="pt").to(device)

for k in [5, 20, 50, 100]:
    print(f"\n--- Top-k: {k} ---")
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,
        top_k=k,
        temperature=0.8,
        pad_token_id=tokenizer.eos_token_id,
    )
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))


--- Top-k: 5 ---
In 36 months, the cheapest place to put AI will be in the top 10.

AI, a popular programming language, is now the top programming language in the world, according to the World Economic Forum. The AI industry is growing at a rapid clip and is expected to grow at an annual growth rate

--- Top-k: 20 ---
In 36 months, the cheapest place to put AI will be in your home, so you'll have more room and less noise.

The best place to place AI is in the middle of the city. That means that you'll have more space to put AI on.

The easiest place to put

--- Top-k: 50 ---
In 36 months, the cheapest place to put AI will be in the lower 48 states. Only four states, including San Francisco - California, Maine, Nebraska and Oregon - have a higher minimum wage.

The higher-paid jobs offer the most benefits: many don't have to put up with the same

--- Top-k: 100 ---
In 36 months, the cheapest place to put AI will be in the top 10.

10. San Francisco

The cost of living in San Francisco 

In [7]:
# Top-p (nucleus) sampling, samples from smallest set of tokens whose cumulative probability >= p
inputs = tokenizer(prompt, return_tensors="pt").to(device)

for p in [0.7, 0.9, 0.95, 0.99]:
    print(f"\n--- Top-p: {p} ---")
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,
        top_p=p,
        temperature=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))


--- Top-p: 0.7 ---
In 36 months, the cheapest place to put AI will be in the UK. In the US, you can buy it in New York, but in China, you can't.

It's a good thing the US isn't so easy to buy AI.

I've spent most of my life

--- Top-p: 0.9 ---
In 36 months, the cheapest place to put AI will be at home.

"If you're really thinking about AI, you can see that we are changing the way we think," said John O'Neill, director of AI at AI North America, an industry research firm.

"There's

--- Top-p: 0.95 ---
In 36 months, the cheapest place to put AI will be at a local, public-owned company, or a local-owned company in the country where they have a monopoly. The majority of AI startups are small- and middle-sized companies. And for those startups, the big money pays.



--- Top-p: 0.99 ---
In 36 months, the cheapest place to put AI will be somewhere else in the top 100 most popular countries per capita. These countries comprise the top 10 of the population who are most likely to choose robot

In [8]:
# Combined best practices (recommended settings)
inputs = tokenizer(prompt, return_tensors="pt").to(device)

outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=True,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    num_return_sequences=3,  # Generate 3 variations
    pad_token_id=tokenizer.eos_token_id,
)

print(f"Generated {len(outputs)} variations:\n")
for i, output in enumerate(outputs):
    print(f"Variation {i+1}:")
    print(tokenizer.decode(output, skip_special_tokens=True))
    print()

Generated 3 variations:

Variation 1:
In 36 months, the cheapest place to put AI will be on the internet. With $20,000 you can earn a basic internet access pass for $5.00.

With $5.00, you can earn a basic internet access pass for $5.00. You don't need to buy a house - you just need to have your phone number on your home.

- you just need to have your phone number on your home. With $10,000 you can earn a basic internet access pass for $5.00.

Variation 2:
In 36 months, the cheapest place to put AI will be on the internet, and in fact, some of the most popular destinations for AI are a few miles away.

It's not just the cheapest place to put AI in the world. The world's most popular destinations for AI are at least as expensive as places for humans, but the world's most expensive place to put AI in the world is just a bit too far away.

The same goes for the next most popular country for AI, China. If you're looking for an extra

Variation 3:
In 36 months, the cheapest place to put AI 

In [9]:
# Batch generation, process multiple prompts at once
prompts = [
    "The availablility of energy is",
    "The output of electricity is",
    "It's always sunny in",
]

inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(device)

outputs = model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_new_tokens=50,
    do_sample=True,
    temperature=0.8,
    top_p=0.95,
    pad_token_id=tokenizer.eos_token_id,
)

print("Batch Results:\n")
for prompt, output in zip(prompts, outputs):
    print(f"{tokenizer.decode(output, skip_special_tokens=True)}")
    print()

A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


Batch Results:

The availablility of energy is only a matter of time, and the fact that it is available without the need for expensive energy sources makes it an ideal energy source for those who are not on the verge of getting into space or in need of a backup generator.

The Space

The output of electricity isA few hours' work is done and the process is repeated each week, at a rate of about five minutes per day. The output of electricity is at its highest point in the year, when it reaches its lowest point.

An average person

It's always sunny inMana. I can't see you wearing a shirt here. I'm in my mid-20s but I love my hair. I can't tell you how happy I am when I see you."

The two hugged before he left,



In [10]:
# Long-form generation
prompt = (
    "Once you start thinking in terms of what percentage of the Sun's power you are harnessing,"
    "you realize you have to go to space. You can't scale very much on Earth."
)
inputs = tokenizer(prompt, return_tensors="pt").to(device)

outputs = model.generate(
    **inputs,
    max_new_tokens=300,
    do_sample=True,
    temperature=0.9,
    top_p=0.95,
    repetition_penalty=1.2,  # Penalize repeated phrases
    pad_token_id=tokenizer.eos_token_id,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Once you start thinking in terms of what percentage of the Sun's power you are harnessing,you realize you have to go to space. You can't scale very much on Earth. So if that is your goal then perhaps consider trying out a different way or using some other means."
Brent Knebel may be reached at bkoenfeld@njadvancemedia . Follow him @bradleykeen_NJ and Find NJ Gadgets On Facebook
